# Lab 08: Reproducible Python Environments with uv

This notebook introduces **reproducible Python environments** using `uv`, the modern Python project and version manager.
The focus is on understanding *why* we isolate and pin environments, and how to make a project that runs identically on any machine.

Installation speed is NOT the goal of this lab. Reproducibility and isolation are.

> **Running this in Colab.** Colab already gives you a fresh, disposable runtime, so the usual "protect my machine" reason for environments is muted here. In Colab, `uv` earns its place differently: it lets you run a **newer Python than Colab ships**, and it lets you *practise the exact commands* you would use in a real terminal. Shell commands are prefixed with `!`, and `%cd` changes the working directory across cells.

---

## Learning Outcomes

By the end of this lab, you should be able to:
1. Explain why isolated environments matter, in terms of dependency conflicts and reproducibility.
2. Install `uv` and use it to install and pin a specific Python version.
3. Create a project that pins both its Python version and its dependencies.
4. Distinguish the files you commit to git (`pyproject.toml`, `uv.lock`, `.python-version`) from the one you never commit (`.venv/`).
5. Reproduce an environment on another machine with a single command.

## 0. Environment Setup

There is nothing to import for this lab — the "environment" *is* the subject. Everything runs through `uv` on the command line.

One thing to fix in your head before starting: in a notebook, a shell command needs a `!`, and a directory change needs `%cd` (a magic), not `!cd`. We will see why in Step 4.

## Step 1: Why environments exist

Installing packages straight into your system Python causes two problems.

**Conflicts**
- Project A needs `numpy 1.26`, Project B needs `numpy 2.1`.
- Only one version can be installed system-wide. One project breaks.

**Reproducibility**
- "It works on my machine" — because your machine has thirty stray packages nobody else has.
- A colleague cannot rebuild what you never wrote down.

A **virtual environment** is a private copy of Python and its packages, living in a folder inside your project. Each project gets its own. `uv` goes one step further: it also installs and pins the **Python version itself**, so you are not at the mercy of whatever Python happens to be on the machine.

**Rule of thumb**

A reproducible project pins three things: the Python version, the packages, and their exact resolved versions.

## Step 2: Install uv

In Colab the reliable install is via pip — it lands straight on the PATH so `uv` is callable immediately.

In [ ]:
!pip install -q uv
!uv --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 68.2 MB/s eta 0:00:00
uv 0.11.21 (x86_64-unknown-linux-gnu)


**Common mistake to highlight!**

On your **own machine** you would not use pip. You would use the standalone installer, which does not depend on any existing Python:

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows PowerShell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

After that you must open a **new** terminal so the updated PATH takes effect — forgetting this is the most common "command not found: uv" error.

## Step 3: Pinning the Python version

This is the part that matters for *this* notebook, and the answer to "can we use a newer Python?".

### Step 3.1: See what Python Colab gives you

Note the version — Colab typically ships a Python a release or two behind the latest.

In [ ]:
!python --version

Python 3.12.13


### Step 3.2: Install Python 3.14 with uv

`uv` downloads and manages Python interpreters itself, in its own folder. This does **not** touch the system Python and needs no admin rights.

In [ ]:
!uv python install 3.14

Installed Python 3.14.6 in 1.79s
 + cpython-3.14.6-linux-x86_64-gnu (python3.14)


### Step 3.3: Run code under Python 3.14

`uv run --python 3.14` runs the command under the version you ask for, fetching it first if needed.

In [ ]:
!uv run --python 3.14 python -c "import sys; print('Running under Python', sys.version.split()[0])"


Running under Python 3.14.6


**Rule of thumb**

Pin the Python version explicitly. Never rely on "whatever Python is installed" — that is exactly what makes results unreproducible.

On an existing project — this is the dedicated command:

```
uv python pin 3.14
```

For a single command only (no persistent pin):

```
uv run --python 3.14 python script.py
```

## Step 4: Create a reproducible project

The same project workflow you would use locally. `uv init` scaffolds a project and pins the Python version; `uv add` resolves packages and writes a lockfile.

### Step 4.1: Initialise the project

In [ ]:
!uv init ml-lab --python 3.14

Initialized project `ml-lab` at `/content/ml-lab`


This creates a small project folder. The files that matter are:
- `pyproject.toml` — project config, including `requires-python = ">=3.14"`
- `.python-version` — the exact version uv will use (`3.14`)
- `.gitignore` — already set up to exclude the environment folder

The `.venv/` environment is **not** created yet — uv builds it lazily on the first `uv add` or `uv run`.

### Step 4.2: Common mistake — `!cd` versus `%cd`

Before we move into the project folder, a trap that catches everyone in notebooks.

Each `!` command runs in its **own fresh shell**, so a directory change with `!cd` is forgotten the instant the cell finishes. To change the working directory for *all later cells*, you must use the `%cd` line magic.

**Exercise 1: Predict the output**

Run the two cells below. Before running, predict: after `!cd ml-lab`, does `!ls` show the *project* contents or the *original* folder?

In [ ]:
!cd ml-lab
!ls

ml-lab	sample_data


### Answer

`!ls` still lists the **original** directory, not `ml-lab`. The `!cd ml-lab` ran in a throwaway shell that exited immediately, so it had no effect on the next cell. Use `%cd` instead — that is what the next cell does, and the change sticks.

In [ ]:
%cd ml-lab

/content/ml-lab


### Step 4.3: Add the packages

Add the libraries the ML lab needs.

In [ ]:
!uv add numpy pandas matplotlib scikit-learn

Using CPython 3.14.6
Creating virtual environment at: .venv
Resolved 19 packages in 580ms
Prepared 17 packages in 3.31s
Installed 17 packages in 116ms
 + contourpy==1.3.3
 + cycler==0.12.1
 + fonttools==4.63.0
 + joblib==1.5.3
 + kiwisolver==1.5.0
 + matplotlib==3.11.0
 + narwhals==2.22.1
 + numpy==2.4.6
 + packaging==26.2
 + pandas==3.0.3
 + pillow==12.2.0
 + pyparsing==3.3.2
 + python-dateutil==2.9.0.post0
 + scikit-learn==1.9.0
 + scipy==1.17.1
 + six==1.17.0
 + threadpoolctl==3.6.0


That single command did three things:
- wrote the packages into `pyproject.toml`
- resolved exact versions into a lockfile, `uv.lock`
- installed everything into `.venv/`

No activation step was needed.

## Step 5: Run code in the project environment

`uv run` executes inside the project's environment automatically. There is no `activate` step to remember or forget.

In [ ]:
!uv run python -c "import sys, sklearn; print('Python', sys.version.split()[0]); print('scikit-learn', sklearn.__version__)"


Python 3.14.6
scikit-learn 1.9.0


**Rule of thumb**

`uv run` replaces `source .venv/bin/activate`. If you are reaching for `activate`, you are probably using the old workflow.

To work in the ML notebook itself, you would add Jupyter and launch it the same way:
```bash
uv add jupyterlab
uv run jupyter lab
```

## Step 6: What to commit (and what never to commit)

Look at the files the project produced.

In [ ]:
!echo '--- pyproject.toml ---'; cat pyproject.toml
!echo; echo '--- .python-version ---'; cat .python-version
!echo; echo '--- .gitignore (note .venv is already ignored) ---'; cat .gitignore

--- pyproject.toml ---
[project]
name = "ml-lab"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.14"
dependencies = [
    "matplotlib>=3.11.0",
    "numpy>=2.4.6",
    "pandas>=3.0.3",
    "scikit-learn>=1.9.0",
]

--- .python-version ---
3.14

--- .gitignore (note .venv is already ignored) ---
# Python-generated files
__pycache__/
*.py[oc]
build/
dist/
wheels/
*.egg-info

# Virtual environments
.venv


**Common mistake to highlight!**

Beginners often commit the entire `.venv/` folder to git. It is large, machine-specific, and pointless to share — and `uv init` already added it to `.gitignore` for you.

You commit the **recipe**, not the **cooked meal**:
- Commit: `pyproject.toml`, `uv.lock`, `.python-version`
- Never commit: `.venv/`

**Rule of thumb**

If a file can be regenerated from `uv sync`, it does not belong in git.

## Step 7: Reproduce the environment anywhere

This is the payoff. Anyone who clones the project rebuilds the *exact* same environment — same Python version, same packages, same resolved versions — with one command:

```bash
uv sync
```

`uv sync` reads `uv.lock` and `.python-version` and recreates `.venv/` to match. This is what a teammate experiences on their first clone, and what you experience on every new Colab runtime.

## Reflection Questions

Why do we commit `uv.lock` but not `.venv/`?

Does installing Python 3.14 with uv change the system Python?

In Colab specifically, what does uv actually buy you?

**Why do we commit `uv.lock` but not `.venv/`?**

`uv.lock` is a small text file that records the exact version of every package, so the environment can be rebuilt identically. `.venv/` is the large, machine-specific *result* of that rebuild — regenerable at any time with `uv sync`, so there is no reason to store or share it.

**Does installing Python 3.14 with uv change the system Python?**

No. uv keeps its interpreters in its own user-owned folder and never touches the system Python or your PATH unless you explicitly opt in. Existing `python` commands keep working.

**In Colab specifically, what does uv actually buy you?**

Two things: the ability to run a **newer Python than Colab ships**, and a safe place to **practise the real commands**. The isolation benefit itself is muted, because Colab's runtime is already disposable.

## Final Conceptual Exercise: Set up the environment for the ML lab

Put the whole workflow together. Imagine you are preparing the environment a classmate will use to run the clustering / PCA / regression notebook.

### Task 1: Initialise a project pinned to Python 3.14

Create a fresh project named `ml-course` pinned to Python 3.14.

In [ ]:
%cd /content
!uv init ml-course --python 3.14
%cd ml-course

/content
Initialized project `ml-course` at `/content/ml-course`
/content/ml-course


### Task 2: Add every package the ML notebook needs

The notebook uses numpy, pandas, matplotlib and scikit-learn, and is run through JupyterLab.

In [ ]:
!uv add numpy pandas matplotlib scikit-learn jupyterlab

Using CPython 3.14.6
Creating virtual environment at: .venv
Resolved 109 packages in 798ms
Prepared 88 packages in 2.54s
Installed 105 packages in 467ms
 + anyio==4.14.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + async-lru==2.3.0
 + attrs==26.1.0
 + babel==2.18.0
 + beautifulsoup4==4.15.0
 + bleach==6.4.0
 + certifi==2026.6.17
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + comm==0.2.3
 + contourpy==1.3.3
 + cycler==0.12.1
 + debugpy==1.8.21
 + decorator==5.3.1
 + defusedxml==0.7.1
 + executing==2.2.1
 + fastjsonschema==2.21.2
 + fonttools==4.63.0
 + fqdn==1.5.1
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.18
 + ipykernel==7.3.0
 + ipython==9.14.1
 + ipython-pygments-lexers==1.1.1
 + isoduration==20.11.0
 + jedi==0.20.0
 + jinja2==3.1.6
 + joblib==1.5.3
 + json5==0.14.0
 + jsonpointer==3.1.1
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + jupyter-builder==1.0.2
 + jupyter-client==8.9.1
 + jupyter-core==5

### Task 3: Confirm the environment

Verify the Python version is 3.14 and that scikit-learn imports, all through the project environment.

In [ ]:
!uv run python -c "import sys, sklearn, pandas, numpy, matplotlib; print('Python:', sys.version.split()[0]); print('scikit-learn:', sklearn.__version__)"


Python: 3.14.6
scikit-learn: 1.9.0


### Task 4: List the three files you would commit

Print the three files that let a classmate reproduce this environment, and confirm `.venv/` is ignored.

In [ ]:
!ls -1 pyproject.toml uv.lock .python-version
!echo '--- .venv ignored? ---'
!grep -q '.venv' .gitignore && echo 'yes, .venv is in .gitignore'

pyproject.toml
.python-version
uv.lock
--- .venv ignored? ---
yes, .venv is in .gitignore


### Answer

The three files to commit are `pyproject.toml`, `uv.lock`, and `.python-version`. Together they pin the dependencies, the exact resolved versions, and the Python version. The `.venv/` folder is regenerated by `uv sync` and stays in `.gitignore`. A classmate clones the repo and runs a single command — `uv sync` — to get an identical environment.

## Appendix: Mapping the old workflow to uv

| Old (`venv` + `pip`)              | New (`uv`)                          |
|-----------------------------------|-------------------------------------|
| `python -m venv .venv`            | created automatically by `uv add`   |
| `source .venv/bin/activate`       | not needed — use `uv run`           |
| `pip install pandas`              | `uv add pandas`                     |
| `pip freeze > requirements.txt`   | `uv.lock` (written automatically)   |
| `pip install -r requirements.txt` | `uv sync`                           |
| (Python version not pinned)       | `.python-version` pins it           |